In [ ]:
# @title Fastest Prepare Environment (fewer frames/higher fps for speed)
!pip install torch==2.6.0 torchvision==0.21.0
%cd /content

!pip install -q torchsde einops diffusers accelerate xformers==0.0.29.post2
!pip install av
!git clone https://github.com/Isi-dev/ComfyUI
%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/Isi-dev/ComfyUI_GGUF.git
%cd /content/ComfyUI/custom_nodes/ComfyUI_GGUF
!pip install -r requirements.txt
%cd /content/ComfyUI
!apt -y install -qq aria2 ffmpeg

useQ6 = False # @param {"type":"boolean"}

if useQ6:
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/city96/Wan2.1-T2V-14B-gguf/resolve/main/wan2.1-t2v-14b-Q6_K.gguf -d /content/ComfyUI/models/unet -o wan2.1-t2v-14b-Q6_K.gguf
else:
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/city96/Wan2.1-T2V-14B-gguf/resolve/main/wan2.1-t2v-14b-Q5_0.gguf -d /content/ComfyUI/models/unet -o wan2.1-t2v-14b-Q5_0.gguf

!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors -d /content/ComfyUI/models/text_encoders -o umt5_xxl_fp8_e4m3fn_scaled.safetensors
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/vae/wan_2.1_vae.safetensors -d /content/ComfyUI/models/vae -o wan_2.1_vae.safetensors

import torch
import numpy as np
from PIL import Image
import gc
import sys
import random
import os
import imageio
import subprocess
from google.colab import files
from IPython.display import display, HTML, Image as IPImage
sys.path.insert(0, '/content/ComfyUI')

from comfy import model_management

from nodes import (
    CheckpointLoaderSimple,
    CLIPLoader,
    CLIPTextEncode,
    VAEDecode,
    VAELoader,
    KSampler,
    UNETLoader
)

from custom_nodes.ComfyUI_GGUF.nodes import UnetLoaderGGUF
from comfy_extras.nodes_model_advanced import ModelSamplingSD3
from comfy_extras.nodes_hunyuan import EmptyHunyuanLatentVideo
from comfy_extras.nodes_images import SaveAnimatedWEBP
from comfy_extras.nodes_video import SaveWEBM

unet_loader = UnetLoaderGGUF()
clip_loader = CLIPLoader()
clip_encode_positive = CLIPTextEncode()
clip_encode_negative = CLIPTextEncode()
vae_loader = VAELoader()
empty_latent_video = EmptyHunyuanLatentVideo()
ksampler = KSampler()
vae_decode = VAEDecode()
save_webp = SaveAnimatedWEBP()
save_webm = SaveWEBM()

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    for obj in list(globals().values()):
        if torch.is_tensor(obj) or (hasattr(obj, "data") and torch.is_tensor(obj.data)):
            del obj
    gc.collect()

def save_as_mp4(images, filename_prefix, fps, output_dir="/content/ComfyUI/output"):
    os.makedirs(output_dir, exist_ok=True)
    output_path = f"{output_dir}/{filename_prefix}.mp4"
    frames = [(img.cpu().numpy() * 255).astype(np.uint8) for img in images]
    with imageio.get_writer(output_path, fps=fps) as writer:
        for frame in frames:
            writer.append_data(frame)
    return output_path

def save_as_webp(images, filename_prefix, fps, quality=90, lossless=False, method=4, output_dir="/content/ComfyUI/output"):
    os.makedirs(output_dir, exist_ok=True)
    output_path = f"{output_dir}/{filename_prefix}.webp"
    frames = [(img.cpu().numpy() * 255).astype(np.uint8) for img in images]
    kwargs = {
        'fps': int(fps),
        'quality': int(quality),
        'lossless': bool(lossless),
        'method': int(method)
    }
    with imageio.get_writer(output_path, format='WEBP', mode='I', **kwargs) as writer:
        for frame in frames:
            writer.append_data(frame)
    return output_path

def save_as_webm(images, filename_prefix, fps, codec="vp9", quality=32, output_dir="/content/ComfyUI/output"):
    os.makedirs(output_dir, exist_ok=True)
    output_path = f"{output_dir}/{filename_prefix}.webm"
    frames = [(img.cpu().numpy() * 255).astype(np.uint8) for img in images]
    kwargs = {
        'fps': int(fps),
        'quality': int(quality),
        'codec': str(codec),
        'output_params': ['-crf', str(int(quality))]
    }
    with imageio.get_writer(output_path, format='FFMPEG', mode='I', **kwargs) as writer:
        for frame in frames:
            writer.append_data(frame)
    return output_path

def save_as_image(image, filename_prefix, output_dir="/content/ComfyUI/output"):
    os.makedirs(output_dir, exist_ok=True)
    output_path = f"{output_dir}/{filename_prefix}.png"
    frame = (image.cpu().numpy() * 255).astype(np.uint8)
    Image.fromarray(frame).save(output_path)
    return output_path

# Minimal frames, high FPS, minimal steps → fastest possible for longer video in about 10 seconds
def generate_video(
    positive_prompt: str = "ultra high res, explicit porn, realistic, naked pornstar, clear genitals, smooth skin, closeup, erotic, creamy skin, big boobs, blowjob, doggy style, creampie, orgasm, cumshot, pov, high contrast, realistic lighting, sharp focus, 8k, NSFW, uncensored",
    negative_prompt: str = "censored, mosaic, watermark, clothed, bad anatomy, poorly rendered face, blurry, cartoon, drawing, bad hands, low quality, covered genitals, ugly, mutated",
    width: int = 512,         # Lowered for fastest generation
    height: int = 576,        # Lowered for fastest generation
    seed: int = 82628696717253,
    steps: int = 12,          # MINIMAL steps for speed
    cfg_scale: float = 1.4,
    sampler_name: str = "uni_pc",
    scheduler: str = "simple",
    frames: int = 8,          # Minimal number of frames so we can use high FPS
    fps: int = 30,            # HIGH FPS to ~simulate a long video
    output_format: str = "mp4"
):
    with torch.inference_mode():
        print(f"Generating video: {frames} frames @ {fps} FPS (effective {frames/fps:.2f}s duration), ultra-fast mode...")
        clip = clip_loader.load_clip("umt5_xxl_fp8_e4m3fn_scaled.safetensors", "wan", "default")[0]
        positive = clip_encode_positive.encode(clip, positive_prompt)[0]
        negative = clip_encode_negative.encode(clip, negative_prompt)[0]
        del clip
        torch.cuda.empty_cache(); gc.collect()
        empty_latent = empty_latent_video.generate(width, height, frames, 1)[0]
        print("Loading Unet Model...")
        if useQ6:
            model = unet_loader.load_unet("wan2.1-t2v-14b-Q6_K.gguf")[0]
        else:
            model = unet_loader.load_unet("wan2.1-t2v-14b-Q5_0.gguf")[0]
        print("Generating video...")
        sampled = ksampler.sample(
            model=model,
            seed=seed,
            steps=steps,
            cfg=cfg_scale,
            sampler_name=sampler_name,
            scheduler=scheduler,
            positive=positive,
            negative=negative,
            latent_image=empty_latent
        )[0]
        del model
        torch.cuda.empty_cache(); gc.collect()
        print("Loading VAE...")
        vae = vae_loader.load_vae("wan_2.1_vae.safetensors")[0]
        try:
            print("Decoding latents...")
            decoded = vae_decode.decode(vae, sampled)[0]
            del vae
            torch.cuda.empty_cache(); gc.collect()
            output_path = ""
            if frames == 1:
                print("Single frame detected - saving as PNG image...")
                output_path = save_as_image(decoded[0], "ComfyUI")
                display(IPImage(filename=output_path))
            else:
                if output_format.lower() == "webm":
                    print("Saving as WEBM...")
                    output_path = save_as_webm(decoded, "ComfyUI", fps, codec="vp9", quality=10)
                elif output_format.lower() == "mp4":
                    print("Saving as MP4...")
                    output_path = save_as_mp4(decoded, "ComfyUI", fps)
                else:
                    raise ValueError(f"Unsupported output format: {output_format}")
                display_video(output_path)
            return output_path
        except Exception as e:
            print(f"Error during decoding/saving: {str(e)}")
            raise
        finally:
            clear_memory()

def display_video(video_path):
    from IPython.display import HTML
    from base64 import b64encode
    video_data = open(video_path,'rb').read()
    if video_path.lower().endswith('.mp4'):
        mime_type = "video/mp4"
    elif video_path.lower().endswith('.webm'):
        mime_type = "video/webm"
    elif video_path.lower().endswith('.webp'):
        mime_type = "image/webp"
    else:
        mime_type = "video/mp4"
    data_url = f"data:{mime_type};base64," + b64encode(video_data).decode()
    display(HTML(f"""
    <video width=512 controls autoplay loop>
        <source src="{data_url}" type="{mime_type}">
    </video>
    """))

print("✅ Environment Setup Complete!")


In [ ]:
# --- Speed-Optimized Gradio UI for 🔥 AI Porn Video Studio ---
import gradio as gr
import os

NSFW_DEFAULT_POSITIVE = (
    "ultra high res, explicit porn, realistic, naked pornstar, clear genitals, smooth skin, closeup, erotic, creamy skin, big boobs, blowjob, doggy style, creampie, orgasm, cumshot, pov, high contrast, realistic lighting, sharp focus, 8k, NSFW, uncensored"
)
NSFW_DEFAULT_NEGATIVE = (
    "censored, mosaic, watermark, clothed, bad anatomy, poorly rendered face, blurry, cartoon, drawing, bad hands, low quality, covered genitals, ugly, mutated"
)

NSFW_PRESETS = {
    "Explicit Doggy": "doggy style sex, penetration, pornographic, naked, creampie, explicit genitals, closeup, realism, pov, high res, uncensored",
    "Hardcore Blowjob": "blowjob, cock in mouth, lips wrapped around dick, explicit, closeup, cum in mouth, saliva, perfect pornstar face, high detail, realism",
    "Lesbian Licking": "lesbian sex, pussy eating, tongue, wet, closeup, orgasm, pornographic, naked, high resolution, sharp focus, uncensored",
    "Riding POV": "cowgirl, riding, bouncing tits, pov, realistic, naked, creampie, porn star, explicit genitals, orgasm, high res, sharp, creamy skin",
    "Ahegao Hentai": "ahegao, intense orgasm, tongue out, drooling, hentai porn, exaggerated face, cum drip, explicit, closeup, high detail",
    "Threesome": "threesome, two girls, one guy, multiple partners, explicit porn, penetration, naked, group sex, cumshot, high res, pov, realism",
    "Anal": "anal sex, explicit, closeup, naked, pov, creampie, perfect ass, high resolution, pornographic, uncensored, realistic"
}

my_css = '''
body {background: #190018;}
.gradio-container {background:#180018;}
.big-header {color: #ff79c6; font-size:2.5em; font-weight:900; margin:0.5em 0 0.2em 0; text-align:center;}
.desc {color:#f8bee0;font-size:1.1em;text-align:center;margin-bottom:1.5em;}
.gr-textbox textarea {font-size:1.119em;}
.gr-button--primary {font-size:1.25em; padding:15px 40px; border-radius:22px; background:linear-gradient(90deg,#ff27b7,#7f33ff);border:none;box-shadow:0 0 24px #a03ccc99;}
.preset-row button {margin:0.15em 0.6em 0.3em 0;padding:10px 23px;border-radius:19px;border:1.15px solid #b138ff77;background:#3a153d;font-weight:700;color:#e1b8f9;font-size:1.03em;}
.preset-row button:active {background:#58218d;}
'''

def run_comfyui(
    pos_prompt,
    neg_prompt,
    frames,
    steps,
    width,
    height,
    use_q6,
    seed,
):
    import random
    global useQ6
    useQ6 = use_q6
    if seed is None or int(seed) < 0:
        seed = random.randint(0, 2**63-1)
    else:
        seed = int(seed)
    # Force minimum settings for fastest output (longer video, ~10s, but fast)
    use_frames = 8   # always minimal for 10s render time
    use_steps = 12   # always low for speed
    use_width = 512  # minimum for speed
    use_height = 576 # minimum for speed
    use_fps = 30     # always high for fake long video
    # Generate the video lightning fast
    video_path = generate_video(
        positive_prompt=str(pos_prompt),
        negative_prompt=str(neg_prompt),
        width=use_width,
        height=use_height,
        frames=use_frames,
        steps=use_steps,
        cfg_scale=1.4,
        sampler_name="uni_pc",
        scheduler="simple",
        fps=use_fps,
        output_format="mp4",
        seed=int(seed)
    )
    if os.path.exists(video_path):
        return video_path
    else:
        raise FileNotFoundError(f"Output video not found: {video_path}")

with gr.Blocks(css=my_css) as demo:
    gr.Markdown("""
    <div class='big-header'>🔥 AI Porn Video Studio</div>
    <div class='desc'>Generate <b>explicit</b> NSFW videos with the latest 14B GGUF T2V AI Model.<br>Ultra-fast: Each video takes only ~10 seconds, longer effect using high FPS. Choose a preset or type your fantasy.<br><b>Note: Adjust FPS/minimal frame count for even faster results.</b></div>
    """)
    with gr.Row():
        positive_box = gr.Textbox(
            label="Positive Prompt",
            value=NSFW_DEFAULT_POSITIVE,
            lines=7,
            max_lines=16,
            elem_classes=["gr-textbox"],
            show_copy_button=True,
            placeholder="Describe your wildest explicit porn scene, e.g.: explicit sex, pornstar, clear genitals, ..."
        )
    gr.Markdown("<b>Quick Presets</b>")
    with gr.Row(elem_classes=["preset-row"]):
        preset_buttons = []
        for lbl, val in NSFW_PRESETS.items():
            btn = gr.Button(lbl)
            preset_buttons.append((btn, val))
    def preset_click(preset_text):
        return gr.update(value=preset_text)
    for btn, val in preset_buttons:
        btn.click(
            fn=lambda x=val: preset_click(x),
            inputs=[],
            outputs=positive_box,
            queue=False
        )
    with gr.Row():
        negative_box = gr.Textbox(
            label="Negative Prompt",
            value=NSFW_DEFAULT_NEGATIVE,
            lines=4,
            max_lines=10,
            elem_classes=["gr-textbox"],
            show_copy_button=True,
            placeholder="What do you want to AVOID? (e.g., censored, watermark, ugly, clothes, ... )"
        )
    gr.Markdown("---")
    gr.Markdown("<b>Advanced Settings (Disabled in Fast Mode)</b> - All videos use minimal frames, high FPS, low resolution for speed.")
    with gr.Row():
        frames = gr.Slider(8, 8, value=8, step=1, label="Frames [Fixed: Fastest]", interactive=False)
        steps = gr.Slider(12, 12, value=12, step=1, label="Sampling Steps [Fixed: Fastest]", interactive=False)
        width = gr.Slider(512, 512, value=512, step=16, label="Width [Fixed]", interactive=False)
        height = gr.Slider(576, 576, value=576, step=16, label="Height [Fixed]", interactive=False)
        use_q6 = gr.Checkbox(label="Use Q6 Model (optional)", value=False)
        seed = gr.Number(label="Seed (-1=random)", value=-1, precision=0)
    gr.Markdown("---")
    with gr.Row():
        gen_btn = gr.Button("💦 Generate", variant="primary")
    output_video = gr.Video(label="AI Porn Video Result (ultra fast)", mirror_webcam=False, elem_id="outvid", interactive=False)
    gen_btn.click(
        run_comfyui,
        inputs=[positive_box, negative_box, frames, steps, width, height, use_q6, seed],
        outputs=output_video,
        show_progress=True,
        api_name=None,
    )
    gr.Markdown("""
    <div style='text-align:center;color:#faf0fa;margin-top:1em;background:#2a1435b3;padding:1em 1em;border-radius:24px;font-size:1em;'>
    <b>TIPS:</b> <b>This mode is speed-optimized:</b> It always produces a high-fps video with minimal frames and low resolution, so every video generates in ~10 seconds regardless of sliders. For best results, use detailed English prompts.<br>
    <b>NSFW ONLY. For SFW, use a different notebook.</b>
    </div>
    """)
    
demo.queue().launch(share=True)
